# Lab: Kokoro ← Stage 1–2 synthesis manifest

**Goal:** Check TTS compatibility with prepared pipeline output (`spoken_text`, `voice_profile_id`, multi-speaker segments).

**Not:** final model selection · production adapter · silent script edits

Runtime: Colab GPU (L4/A100 preferred).

## 0. Config — set your repo URL

In [1]:
# @title Repo settings
REPO_URL = "https://github.com/phamtu2x5/ielts-script2audio.git"
BRANCH = "main"
WORKDIR = "/content/ielts-script2audio"


## 1. Clone repo

In [2]:
import os
from pathlib import Path

if not Path(WORKDIR).exists():
    !git clone --branch {BRANCH} {REPO_URL} {WORKDIR}
else:
    %cd {WORKDIR}
    !git pull
%cd {WORKDIR}
print("CWD:", os.getcwd())

Cloning into '/content/ielts-script2audio'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 86 (delta 7), reused 85 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 98.24 KiB | 10.92 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/ielts-script2audio
CWD: /content/ielts-script2audio


## 2. Install control-plane + Kokoro

Stage 1–2 package is local editable install. Kokoro needs `espeak-ng`.

In [3]:
!pip -q install -e ".[dev]" kokoro soundfile
!apt-get -qq update && apt-get -qq install -y espeak-ng

import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 125.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.5/69.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 3. Stage 1–2: build synthesis manifest

This is the **same contract** production will use later.

In [4]:
!mkdir -p outputs lab_audio/kokoro_part1
!ielts-s2a prepare examples/part1_valid.json --pretty -o outputs/part1_manifest.json

import json
from pathlib import Path

m = json.loads(Path("outputs/part1_manifest.json").read_text())
print("transcript_id:", m["transcript_id"])
print("valid:", m["validation"]["valid"])
print("n_requests:", len(m["requests"]))
print("voice profiles:", [(e["speaker_id"], e["voice_profile_id"]) for e in m["speaker_map"]["entries"]])
for u in m["prepared_utterances"]:
    if u["event_id"] in {"e004", "e006"}:
        print(u["event_id"], "display:", u["display_text"])
        print("       spoken :", u["spoken_text"])

transcript_id: ex-part1-valid-001
valid: True
n_requests: 6
voice profiles: [('spk_a', 'vp_spk_a'), ('spk_b', 'vp_spk_b')]
e004 display: It's Sarah Thompson. That's T-H-O-M-P-S-O-N.
       spoken : It's Sarah Thompson. That's T H O M P S O N.
e006 display: SW1A 1AA.
       spoken : S W one A one A A.


## 4. Lab render (Kokoro consumes manifest)

Maps `voice_profile_id` → Kokoro fixed IDs via `configs/voice_maps/kokoro_en_gb_part1.json`.

Uses **`spoken_text`** (Stage 2), not raw script rewrite.

In [5]:
import json
from pathlib import Path

!python scripts/lab_render_kokoro_from_manifest.py \
  --manifest outputs/part1_manifest.json \
  --voice-map configs/voice_maps/kokoro_en_gb_part1.json \
  --out-dir lab_audio/kokoro_part1

report = json.loads(Path("lab_audio/kokoro_part1/lab_render_report.json").read_text())
print("ok:", report["ok_count"], "fail:", report["fail_count"])
assert report["ok_count"] > 0, "No segments rendered — fix install/render before tracking"
for s in report["segments"]:
    print(s["segment_id"], s["status"], s.get("backend_voice_id"), s.get("error") or "")


config.json: 100% 2.35k/2.35k [00:00<00:00, 9.93MB/s]
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)

kokoro-v1_0.pth: downloading bytes:  73% 238M/327M [00:02<00:00, 258MB/s, 18.7MB/s  ]
kokoro-v1_0.pth: downloading bytes:  85% 277M/327M [00:02<00:00, 190MB/s, 24.6MB/s  ]
kokoro-v1_0.pth: downloading bytes:  94% 307M/327M [00:02<00:00, 177MB/s, 26.0MB/s  ]
kokoro-v1_0.pth: downloading bytes: 100% 309M/309M [00:02<00:00, 111MB/s, 27.6MB/s  ]
kokoro-v1_0.pth: reconstructing file: 100% 327M/327M [00:02<00:00, 117MB/s, 30.2MB/s  ]


## 5. Tracking (một cách duy nhất)

Với **từng** segment theo thứ tự:

1. Đọc **DISPLAY** (script gốc) và **SPOKEN** (text đã đưa TTS).
2. Bấm play audio ngay bên dưới.
3. Ghi vào dict `reviews` ở cell sau: `yes` / `partial` / `no` + notes nếu cần.

Không dùng thêm CSV/HTML/terminal trong notebook này — chỉ vòng lặp nghe + dict `reviews`.


In [ ]:
from pathlib import Path
from IPython.display import Audio, display
import json

REPORT_PATH = Path("lab_audio/kokoro_part1/lab_render_report.json")
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
segments = report.get("segments") or []
audio_dir = REPORT_PATH.parent

print(f"Transcript: {report.get('transcript_id')} | segments: {len(segments)}")
print(f"Text field used by TTS: {report.get('text_field_used')}")
print()

for i, seg in enumerate(segments, start=1):
    sid = seg.get("segment_id")
    fname = seg.get("output_filename") or (Path(seg["output"]).name if seg.get("output") else None)
    wav = audio_dir / fname if fname else None

    print("=" * 72)
    print(f"[{i}/{len(segments)}] {sid} | speaker={seg.get('speaker_id')} | voice={seg.get('backend_voice_id')} | status={seg.get('status')}")
    print(f"DISPLAY : {seg.get('display_text', '')}")
    print(f"SPOKEN  : {seg.get('spoken_text', '')}")
    if seg.get("protected_region_ids"):
        print(f"protected: {seg.get('protected_region_ids')}")
    if seg.get("error"):
        print(f"ERROR   : {seg.get('error')}")
    if wav is not None and wav.is_file():
        display(Audio(filename=str(wav)))
    else:
        print(f"(missing wav: {fname})")
    print()


## 5b. Ghi kết quả tracking

Chỉ **một** chỗ ghi nhận xét: dict `reviews` bên dưới.

- Key = `segment_id` (nhìn ở cell trên, ví dụ `seg_0001`)
- `content_match`: `yes` | `partial` | `no`
- `notes`: ngắn gọn (để trống nếu không cần)

Chạy cell để lưu `lab_audio/kokoro_part1/segment_review_filled.json`.


In [ ]:
from pathlib import Path
import json

report_path = Path("lab_audio/kokoro_part1/lab_render_report.json")
report = json.loads(report_path.read_text(encoding="utf-8"))

# Pre-fill one entry per rendered segment — only edit content_match / notes
reviews = {
    seg["segment_id"]: {"content_match": "", "notes": ""}
    for seg in (report.get("segments") or [])
    if seg.get("segment_id")
}

# Example after listening:
# reviews["seg_0004"]["content_match"] = "partial"
# reviews["seg_0004"]["notes"] = "spelling still a bit fast"

filled = []
missing = []
for seg in report.get("segments") or []:
    sid = seg.get("segment_id")
    human = reviews.get(sid) or {}
    match = (human.get("content_match") or "").strip().lower()
    if match not in {"yes", "partial", "no"}:
        missing.append(sid)
    filled.append(
        {
            "segment_id": sid,
            "event_id": seg.get("event_id"),
            "speaker_id": seg.get("speaker_id"),
            "backend_voice_id": seg.get("backend_voice_id"),
            "output_filename": seg.get("output_filename"),
            "display_text": seg.get("display_text"),
            "spoken_text": seg.get("spoken_text"),
            "status": seg.get("status"),
            "content_match": match,
            "notes": human.get("notes", ""),
        }
    )

out = Path("lab_audio/kokoro_part1/segment_review_filled.json")
out.write_text(json.dumps(filled, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"Wrote {out}")
if missing:
    print("Chưa chấm (trống/invalid):", ", ".join(missing))
else:
    print("Đã chấm đủ", len(filled), "segment.")
for row in filled:
    print(f"{row['segment_id']}: match={row['content_match'] or '(empty)'} | {(row['display_text'] or '')[:60]}")


## 6. Ghép một file nghe liền (tuỳ chọn)

Mặc định dùng **khoảng lặng thích ứng** (không còn 450ms cứng cho mọi chỗ):

- cùng speaker nói tiếp → nghỉ ngắn hơn
- đổi speaker (lượt thoại) → nghỉ dài hơn
- câu trước dài → thêm chút “thở”

Chạy cell dưới để tạo `part1_full.wav` và nghe.


In [ ]:
import json
from pathlib import Path
from IPython.display import Audio, display

# adaptive gaps (recommended)
!python scripts/lab_concat_segment_wavs.py \
  --report lab_audio/kokoro_part1/lab_render_report.json \
  --out lab_audio/kokoro_part1/part1_full.wav \
  --gap-mode adaptive \
  --same-speaker-gap-ms 220 \
  --turn-gap-ms 520

# Optional: fixed gap (old behaviour)
# !python scripts/lab_concat_segment_wavs.py \
#   --report lab_audio/kokoro_part1/lab_render_report.json \
#   --out lab_audio/kokoro_part1/part1_full_fixed.wav \
#   --gap-mode fixed --gap-ms 450

full = Path("lab_audio/kokoro_part1/part1_full.wav")
meta_path = Path("lab_audio/kokoro_part1/part1_full.concat.json")
meta = json.loads(meta_path.read_text())
print("duration_sec:", meta.get("duration_sec"), "segments:", meta.get("num_segments_used"))
print("gap sample:", meta.get("gaps", [])[:5])
if full.is_file():
    display(Audio(filename=str(full)))


## 7. Checklist nhanh (lab)

Sau khi nghe từng câu (§5) và/hoặc file ghép (§6):

1. Hai speaker có hai giọng khác nhau không?
2. Spelling / postcode / phone có **đủ chậm để bắt** không?
3. Có crash / thiếu wav không?
4. Có phải sửa script mới chạy được không? (**Không** silent-fix.)

**Verdict lab:** Pass / Borderline / Fail — **chưa** chọn TTS cuối.
